In [ ]:
from pathlib import Path
import importlib.util
build_dir = Path("build")
module_path = next(build_dir.glob("simplinho*.so"), None)
print('module_path', module_path)
if module_path is None:
    raise FileNotFoundError('Cannot locate simplinho extension module in build directory')


In [ ]:
import sys
from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd

# Load the built simplinho extension from build directory
build_dir = Path("build")
module_path = next(build_dir.glob("simplinho*.so"), None)
if module_path is None:
    raise FileNotFoundError("Could not find simplinho extension module in build directory")
spec = importlib.util.spec_from_file_location("simplinho", module_path)
simplinho = importlib.util.module_from_spec(spec)
spec.loader.exec_module(simplinho)
print("Loaded simplinho module from", module_path)
print("Python executable:", sys.executable)
print("simplinho objects:", hasattr(simplinho, "Model"), hasattr(simplinho, "BranchAndBoundOptions"))

n = 100
profits = []
weights1 = []
weights2 = []
weights3 = []
groups = []
for i in range(n):
    profit = float(((17 * i + 13) % 97) + 10)
    w1 = float(((11 * i + 7) % 29) + 1)
    w2 = float(((19 * i + 5) % 31) + 1)
    w3 = float(((23 * i + 3) % 37) + 1)
    group = 1.0 if i % 10 in (0, 1, 2, 3) else 0.0
    profits.append(profit)
    weights1.append(w1)
    weights2.append(w2)
    weights3.append(w3)
    groups.append(group)
constraints = [
    (weights1, "<=", 0.35 * sum(weights1), "cap_1"),
    (weights2, "<=", 0.33 * sum(weights2), "cap_2"),
    (weights3, "<=", 0.31 * sum(weights3), "cap_3"),
    (groups, "<=", max(8.0, 0.18 * n), "group_cap"),
]
model = simplinho.Model()
x_vars = []
for i in range(n):
    x = model.add_binary_var(f"x_{i}", obj=profits[i])
    x_vars.append(x)
for coeffs, sense, rhs, name in constraints:
    expr = 0.0
    for i, coeff in enumerate(coeffs):
        expr = expr + coeff * x_vars[i]
    if sense == "<=":
        model.add_constr(expr <= rhs, name=name)
    elif sense == ">=":
        model.add_constr(expr >= rhs, name=name)
    else:
        model.add_constr(expr == rhs, name=name)
model.maximize(sum(profits[i] * x_vars[i] for i in range(n)))
print("Model built: vars=", n, "constraints=", len(constraints))

options = simplinho.BranchAndBoundOptions()
options.max_nodes = 10000
options.node_selection = simplinho.NodeSelectionStrategy.BestBound
options.parallel_workers = 1
options.use_async_heuristics = False
options.use_cut_pool = False
options.use_gomory_cuts = False
options.use_cover_cuts = False
options.use_feasibility_pump = False
options.use_rens = False
options.use_rins = False
options.use_local_search = False
options.use_local_branching = False

result = model.solve_mip(options)
print("Simplinho status:", result.status)
print("Objective:", result.obj)
print("Best bound:", result.best_bound)
print("Node count:", result.node_count)
print("Root relaxation objective:", result.root_relaxation_objective)
print("Has solution:", result.has_solution)
print("Warm start used:", result.warm_start_basis_state_used)
if result.has_solution:
    x_vals = [result.value(v) for v in x_vars]
    print("First 10 x:", x_vals[:10])
else:
    print("No incumbent found")
print("Tree nodes:", len(result.tree_nodes))
for node in result.tree_nodes[:10]:
    print(node.id, node.parent_id, node.depth, node.status, node.branch_var, node.branch_value, node.bound)

import pulp
pulp_model = pulp.LpProblem("binary_multiknapsack", pulp.LpMaximize)
pulp_x = [pulp.LpVariable(f"x_{i}", cat="Binary") for i in range(n)]
pulp_model += pulp.lpSum(profits[i] * pulp_x[i] for i in range(n))
pulp_model += pulp.lpSum(weights1[i] * pulp_x[i] for i in range(n)) <= 0.35 * sum(weights1)
pulp_model += pulp.lpSum(weights2[i] * pulp_x[i] for i in range(n)) <= 0.33 * sum(weights2)
pulp_model += pulp.lpSum(weights3[i] * pulp_x[i] for i in range(n)) <= 0.31 * sum(weights3)
pulp_model += pulp.lpSum(groups[i] * pulp_x[i] for i in range(n)) <= max(8.0, 0.18 * n)
pulp_status = pulp_model.solve(pulp.PULP_CBC_CMD(msg=False))
print("PuLP status:", pulp.LpStatus[pulp_status])
print("PuLP objective:", pulp.value(pulp_model.objective))
pulp_vals = [v.value() for v in pulp_x]
print("PuLP first 10 x:", pulp_vals[:10])

import pandas as pd
diag = {
    "simplinho_status": result.status,
    "simplinho_obj": result.obj,
    "simplinho_best_bound": result.best_bound,
    "simplinho_has_solution": result.has_solution,
    "simplinho_node_count": result.node_count,
    "pulp_status": pulp.LpStatus[pulp_status],
    "pulp_obj": pulp.value(pulp_model.objective),
}
print(pd.Series(diag))

In [ ]:
# Inspect direct LP relaxations for the root and fixed child cases
from scipy.sparse import csr_matrix
from math import ceil, floor
from typing import Optional

# Build low-level LP matrices from the same problem
n = 100
A = np.zeros((4, n))
b = np.zeros(4)
for i in range(n):
    A[0, i] = weights1[i]
    A[1, i] = weights2[i]
    A[2, i] = weights3[i]
    A[3, i] = groups[i]
b[0] = 0.35 * sum(weights1)
b[1] = 0.33 * sum(weights2)
b[2] = 0.31 * sum(weights3)
b[3] = max(8.0, 0.18 * n)
c = np.array(profits)
l = np.zeros(n)
u = np.ones(n)
solver = simplinho.RevisedSimplex()
# Try root relaxation
root = solver.solve(A, b, c, l, u)
print('root status', root.status, 'obj', root.obj)
# Try children fixed cases
for fix_val in [0.0, 1.0]:
    l_fix = l.copy()
    u_fix = u.copy()
    var = 54
    l_fix[var] = fix_val
    u_fix[var] = fix_val
    try:
        child = solver.solve(A, b, c, l_fix, u_fix)
        print('fixed', var, fix_val, 'status', child.status, 'obj', child.obj)
    except Exception as exc:
        print('fixed', var, fix_val, 'exception', exc)


In [ ]:
from scipy.optimize import linprog
c_neg = -np.array(profits)  # maximize c^T x -> minimize -c^T x
A_ub = np.vstack([weights1, weights2, weights3, groups])
b_ub = np.array([0.35 * sum(weights1), 0.33 * sum(weights2), 0.31 * sum(weights3), max(8.0, 0.18 * n)])
bounds = [(0.0, 1.0)] * n
for fix_val in [None, 0.0, 1.0]:
    if fix_val is None:
        res = linprog(c_neg, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
        label = 'root'
    else:
        bnds = bounds.copy()
        bnds[54] = (fix_val, fix_val)
        res = linprog(c_neg, A_ub=A_ub, b_ub=b_ub, bounds=bnds, method='highs')
        label = f'fixed_54_{fix_val}'
    print(label, res.success, res.status, res.fun if res.success else None, 'x54', res.x[54] if res.success else None)
    if hasattr(res,'message'):
        print(' message:', res.message)


In [ ]:
print([name for name in dir(simplinho) if 'presolve' in name.lower() or 'root' in name.lower() or 'node' in name.lower()][:50])


In [ ]:
# Build the LP with slack variables and test warm-start solves
n = 100
num_constraints = 4
total_vars = n + num_constraints
A = np.zeros((num_constraints, total_vars))
b = np.zeros(num_constraints)
c = np.zeros(total_vars)
l = np.zeros(total_vars)
u = np.full(total_vars, np.inf)
for i in range(n):
    A[0, i] = weights1[i]
    A[1, i] = weights2[i]
    A[2, i] = weights3[i]
    A[3, i] = groups[i]
b[0] = 0.35 * sum(weights1)
b[1] = 0.33 * sum(weights2)
b[2] = 0.31 * sum(weights3)
b[3] = max(8.0, 0.18 * n)
for row in range(num_constraints):
    slack = n + row
    A[row, slack] = 1.0
    l[slack] = 0.0
    u[slack] = np.inf
c[:n] = -np.array(profits)  # solver is min c^T x, original maximize -> c negative
solver = simplinho.RevisedSimplex()
root = solver.solve(A, b, c, l, u)
print('root status', root.status, 'obj', root.obj)
print('root x54', root.x[54])
print('root has basis', hasattr(root, 'basis_state'), 'basis len', len(root.basis_state.column_status))
basis = root.basis_state if root.status == simplinho.LPSolution.Status.Optimal else None
for fix_val in [0.0, 1.0]:
    l_fix = l.copy()
    u_fix = u.copy()
    l_fix[54] = fix_val
    u_fix[54] = fix_val
    if basis is not None:
        child = solver.solve(A, b, c, l_fix, u_fix, basis)
    else:
        child = solver.solve(A, b, c, l_fix, u_fix)
    print('child fixed', fix_val, 'status', child.status, 'obj', child.obj)


In [ ]:
import importlib
spec = importlib.util.spec_from_file_location("simplinho", module_path)
simplinho = importlib.util.module_from_spec(spec)
spec.loader.exec_module(simplinho)
print('reloaded simplinho module')
n = 100
profits = [float(((17 * i + 13) % 97) + 10) for i in range(n)]
weights1 = [float(((11 * i + 7) % 29) + 1) for i in range(n)]
weights2 = [float(((19 * i + 5) % 31) + 1) for i in range(n)]
weights3 = [float(((23 * i + 3) % 37) + 1) for i in range(n)]
groups = [1.0 if i % 10 in (0, 1, 2, 3) else 0.0 for i in range(n)]
model = simplinho.Model()
x_vars = [model.add_binary_var(f'x_{i}', obj=profits[i]) for i in range(n)]
constraints = [
    (weights1, '<=', 0.35 * sum(weights1), 'cap_1'),
    (weights2, '<=', 0.33 * sum(weights2), 'cap_2'),
    (weights3, '<=', 0.31 * sum(weights3), 'cap_3'),
    (groups, '<=', max(8.0, 0.18 * n), 'group_cap'),
 ]
for coeffs, sense, rhs, name in constraints:
    expr = 0.0
    for i, coeff in enumerate(coeffs):
        expr = expr + coeff * x_vars[i]
    if sense == '<=':
        model.add_constr(expr <= rhs, name=name)
    elif sense == '>=':
        model.add_constr(expr >= rhs, name=name)
    else:
        model.add_constr(expr == rhs, name=name)
model.maximize(sum(profits[i] * x_vars[i] for i in range(n)))
options = simplinho.BranchAndBoundOptions()
options.max_nodes = 10000
options.node_selection = simplinho.NodeSelectionStrategy.BestBound
options.parallel_workers = 1
options.use_async_heuristics = False
options.use_cut_pool = False
options.use_gomory_cuts = False
options.use_cover_cuts = False
options.use_feasibility_pump = False
options.use_rens = False
options.use_rins = False
options.use_local_search = False
options.use_local_branching = False
result = model.solve_mip(options)
print('Simplinho status:', result.status)
print('Objective:', result.obj)
print('Best bound:', result.best_bound)
print('Node count:', result.node_count)
print('Root relaxation objective:', result.root_relaxation_objective)
print('Has solution:', result.has_solution)
print('Warm start used:', result.warm_start_basis_state_used)
print('Tree nodes:', len(result.tree_nodes))
for node in result.tree_nodes:
    print(node.id, node.parent_id, node.depth, node.status, node.branch_var, node.branch_value, node.bound)


In [ ]:
import importlib
spec = importlib.util.spec_from_file_location("simplinho", module_path)
simplinho = importlib.util.module_from_spec(spec)
spec.loader.exec_module(simplinho)
print('reloaded simplinho module for verbose debug')
n = 100
profits = [float(((17 * i + 13) % 97) + 10) for i in range(n)]
weights1 = [float(((11 * i + 7) % 29) + 1) for i in range(n)]
weights2 = [float(((19 * i + 5) % 31) + 1) for i in range(n)]
weights3 = [float(((23 * i + 3) % 37) + 1) for i in range(n)]
groups = [1.0 if i % 10 in (0, 1, 2, 3) else 0.0 for i in range(n)]
model = simplinho.Model()
x_vars = [model.add_binary_var(f'x_{i}', obj=profits[i]) for i in range(n)]
constraints = [
    (weights1, '<=', 0.35 * sum(weights1), 'cap_1'),
    (weights2, '<=', 0.33 * sum(weights2), 'cap_2'),
    (weights3, '<=', 0.31 * sum(weights3), 'cap_3'),
    (groups, '<=', max(8.0, 0.18 * n), 'group_cap'),
 ]
for coeffs, sense, rhs, name in constraints:
    expr = 0.0
    for i, coeff in enumerate(coeffs):
        expr = expr + coeff * x_vars[i]
    if sense == '<=':
        model.add_constr(expr <= rhs, name=name)
    elif sense == '>=':
        model.add_constr(expr >= rhs, name=name)
    else:
        model.add_constr(expr == rhs, name=name)
model.maximize(sum(profits[i] * x_vars[i] for i in range(n)))
options = simplinho.BranchAndBoundOptions()
options.max_nodes = 10000
options.node_selection = simplinho.NodeSelectionStrategy.BestBound
options.parallel_workers = 1
options.use_async_heuristics = False
options.use_cut_pool = False
options.use_gomory_cuts = False
options.use_cover_cuts = False
options.use_feasibility_pump = False
options.use_rens = False
options.use_rins = False
options.use_local_search = False
options.use_local_branching = False
options.verbose = True
result = model.solve_mip(options)
print('Simplinho status:', result.status)
print('Objective:', result.obj)
print('Best bound:', result.best_bound)
print('Node count:', result.node_count)
print('Root relaxation objective:', result.root_relaxation_objective)
print('Has solution:', result.has_solution)
print('Warm start used:', result.warm_start_basis_state_used)


In [ ]:
from scipy.sparse import csr_matrix
import importlib
spec = importlib.util.spec_from_file_location("simplinho", module_path)
simplinho = importlib.util.module_from_spec(spec)
spec.loader.exec_module(simplinho)
n = 100
num_constraints = 4
total_vars = n + num_constraints
A = np.zeros((num_constraints, total_vars))
b = np.zeros(num_constraints)
c = np.zeros(total_vars)
l = np.zeros(total_vars)
u = np.full(total_vars, np.inf)
for i in range(n):
    A[0, i] = weights1[i]
    A[1, i] = weights2[i]
    A[2, i] = weights3[i]
    A[3, i] = groups[i]
b[0] = 0.35 * sum(weights1)
b[1] = 0.33 * sum(weights2)
b[2] = 0.31 * sum(weights3)
b[3] = max(8.0, 0.18 * n)
for row in range(num_constraints):
    A[row, n + row] = 1.0
c[:n] = -np.array(profits)
A_sparse = csr_matrix(A)
for mode in [simplinho.SimplexMode.Auto, simplinho.SimplexMode.Dual, simplinho.SimplexMode.Primal]:
    opt = simplinho.RevisedSimplexOptions()
    opt.mode = mode
    solver = simplinho.RevisedSimplex(opt)
    try:
        root = solver.solve(A_sparse, b, c, l, u)
        print('mode', mode, 'status', root.status, 'obj', root.obj)
    except Exception as e:
        print('mode', mode, 'error', type(e).__name__, e)


In [ ]:
from scipy.sparse import csr_matrix
import importlib
spec = importlib.util.spec_from_file_location("simplinho", module_path)
simplinho = importlib.util.module_from_spec(spec)
spec.loader.exec_module(simplinho)
n = 100
num_constraints = 4
total_vars = n + num_constraints
A = np.zeros((num_constraints, total_vars))
b = np.zeros(num_constraints)
c = np.zeros(total_vars)
l = np.zeros(total_vars)
u = np.full(total_vars, np.inf)
for i in range(n):
    A[0, i] = weights1[i]
    A[1, i] = weights2[i]
    A[2, i] = weights3[i]
    A[3, i] = groups[i]
b[0] = 0.35 * sum(weights1)
b[1] = 0.33 * sum(weights2)
b[2] = 0.31 * sum(weights3)
b[3] = max(8.0, 0.18 * n)
for row in range(num_constraints):
    A[row, n + row] = 1.0
c[:n] = -np.array(profits)
A_sparse = csr_matrix(A)
def try_solve(opt, A_mat, l_vec, u_vec):
    solver = simplinho.RevisedSimplex(opt)
    try:
        sol = solver.solve(A_mat, b, c, l_vec, u_vec)
        return sol
    except Exception as e:
        return e

for fix_val in [0.0, 1.0]:
    l_fix = l.copy(); u_fix = u.copy(); l_fix[54] = fix_val; u_fix[54] = fix_val
    print('=== fixed', fix_val, '===')
    for mode in [simplinho.SimplexMode.Auto, simplinho.SimplexMode.Dual, simplinho.SimplexMode.Primal]:
        opt = simplinho.RevisedSimplexOptions()
        opt.mode = mode
        opt.pricing_rule = 'adaptive'
        res = try_solve(opt, A_sparse, l_fix, u_fix)
        print('mode', mode, 'res', type(res).__name__, getattr(res, 'status', None), getattr(res, 'obj', None))
        if isinstance(res, object) and not isinstance(res, Exception):
            break


In [ ]:
from scipy.sparse import csr_matrix
import importlib
spec = importlib.util.spec_from_file_location("simplinho", module_path)
simplinho = importlib.util.module_from_spec(spec)
spec.loader.exec_module(simplinho)
n = 100
num_constraints = 4
total_vars = n + num_constraints
A = np.zeros((num_constraints, total_vars))
b = np.zeros(num_constraints)
c = np.zeros(total_vars)
l = np.zeros(total_vars)
u = np.full(total_vars, np.inf)
for i in range(n):
    A[0, i] = weights1[i]
    A[1, i] = weights2[i]
    A[2, i] = weights3[i]
    A[3, i] = groups[i]
b[0] = 0.35 * sum(weights1)
b[1] = 0.33 * sum(weights2)
b[2] = 0.31 * sum(weights3)
b[3] = max(8.0, 0.18 * n)
for row in range(num_constraints):
    A[row, n + row] = 1.0
c[:n] = -np.array(profits)
A_sparse = csr_matrix(A)
modes = [simplinho.SimplexMode.Auto, simplinho.SimplexMode.Dual, simplinho.SimplexMode.Primal]
pricing = ['adaptive', 'devex', 'most_negative']
for fix_val in [None, 0.0, 1.0]:
    if fix_val is None:
        print('=== root ===')
        l_vec, u_vec = l, u
    else:
        print('=== fixed', fix_val, '===')
        l_vec = l.copy(); u_vec = u.copy(); l_vec[54] = fix_val; u_vec[54] = fix_val
    for mode in modes:
        for p in pricing:
            opt = simplinho.RevisedSimplexOptions()
            opt.mode = mode
            opt.pricing_rule = p
            solver = simplinho.RevisedSimplex(opt)
            try:
                sol = solver.solve(A_sparse, b, c, l_vec, u_vec)
                print('mode', mode, 'pricing', p, 'status', sol.status, 'obj', sol.obj)
            except Exception as e:
                print('mode', mode, 'pricing', p, 'exception', type(e).__name__, e)


In [ ]:
from scipy.sparse import csr_matrix
import importlib
spec = importlib.util.spec_from_file_location("simplinho", module_path)
simplinho = importlib.util.module_from_spec(spec)
spec.loader.exec_module(simplinho)
n = 100
num_constraints = 4
total_vars = n + num_constraints
A = np.zeros((num_constraints, total_vars))
b = np.zeros(num_constraints)
c = np.zeros(total_vars)
l = np.zeros(total_vars)
u = np.full(total_vars, np.inf)
for i in range(n):
    A[0, i] = weights1[i]
    A[1, i] = weights2[i]
    A[2, i] = weights3[i]
    A[3, i] = groups[i]
b[0] = 0.35 * sum(weights1)
b[1] = 0.33 * sum(weights2)
b[2] = 0.31 * sum(weights3)
b[3] = max(8.0, 0.18 * n)
for row in range(num_constraints):
    A[row, n + row] = 1.0
c[:n] = -np.array(profits)
A_sparse = csr_matrix(A)
def try_solve(opt, A_mat, l_vec, u_vec):
    solver = simplinho.RevisedSimplex(opt)
    try:
        sol = solver.solve(A_mat, b, c, l_vec, u_vec)
        return sol
    except Exception as e:
        return e

for fix_val in [0.0, 1.0]:
    l_fix = l.copy(); u_fix = u.copy(); l_fix[54] = fix_val; u_fix[54] = fix_val
    print('=== fixed', fix_val, '===')
    modes = [simplinho.SimplexMode.Auto, simplinho.SimplexMode.Dual, simplinho.SimplexMode.Primal]
    for mode in modes:
        opt = simplinho.RevisedSimplexOptions()
        opt.mode = mode
        opt.pricing_rule = 'adaptive'
        res = try_solve(opt, A_sparse, l_fix, u_fix)
        if isinstance(res, Exception):
            print('mode', mode, 'exception', type(res).__name__, res)
        else:
            print('mode', mode, 'status', res.status, 'obj', res.obj)
            if res.status == simplinho.LPSolution.Status.Optimal:
                break
    print('--- fallback test ---')
    if isinstance(res, Exception) or res.status != simplinho.LPSolution.Status.Optimal:
        for fallback_mode in modes:
            if fallback_mode == modes[0]:
                continue
            opt = simplinho.RevisedSimplexOptions()
            opt.mode = fallback_mode
            opt.pricing_rule = 'devex'
            res2 = try_solve(opt, A_sparse, l_fix, u_fix)
            print('fallback', fallback_mode, 'status', getattr(res2, 'status', type(res2).__name__), getattr(res2, 'obj', None))
            if not isinstance(res2, Exception) and res2.status == simplinho.LPSolution.Status.Optimal:
                break


In [ ]:
from scipy.sparse import csr_matrix
import importlib
spec = importlib.util.spec_from_file_location("simplinho", module_path)
simplinho = importlib.util.module_from_spec(spec)
spec.loader.exec_module(simplinho)
n = 100
num_constraints = 4
total_vars = n + num_constraints
A = np.zeros((num_constraints, total_vars))
b = np.zeros(num_constraints)
c = np.zeros(total_vars)
l = np.zeros(total_vars)
u = np.full(total_vars, np.inf)
for i in range(n):
    A[0, i] = weights1[i]
    A[1, i] = weights2[i]
    A[2, i] = weights3[i]
    A[3, i] = groups[i]
b[0] = 0.35 * sum(weights1)
b[1] = 0.33 * sum(weights2)
b[2] = 0.31 * sum(weights3)
b[3] = max(8.0, 0.18 * n)
for row in range(num_constraints):
    A[row, n + row] = 1.0
c[:n] = -np.array(profits)
A_sparse = csr_matrix(A)
scenarios = [
    {'mode': simplinho.SimplexMode.Auto, 'pricing_rule': 'adaptive'},
    {'mode': simplinho.SimplexMode.Auto, 'pricing_rule': 'devex'},
    {'mode': simplinho.SimplexMode.Auto, 'pricing_rule': 'most_negative'},
    {'mode': simplinho.SimplexMode.Auto, 'pricing_rule': 'devex', 'partial_pricing': True},
    {'mode': simplinho.SimplexMode.Auto, 'pricing_rule': 'devex', 'dual_pricing': 'switch'},
    {'mode': simplinho.SimplexMode.Auto, 'pricing_rule': 'devex', 'basis_update': 'eta'},
    {'mode': simplinho.SimplexMode.Dual, 'pricing_rule': 'devex', 'dual_pricing': 'switch'},
    {'mode': simplinho.SimplexMode.Primal, 'pricing_rule': 'devex', 'partial_pricing': True},
    {'mode': simplinho.SimplexMode.Primal, 'pricing_rule': 'most_negative', 'basis_update': 'eta'},
    {'mode': simplinho.SimplexMode.Dual, 'pricing_rule': 'devex', 'partial_pricing': True},
 ]
for fix_val in [None, 0.0, 1.0]:
    if fix_val is None:
        print('=== root ===')
        l_vec, u_vec = l, u
    else:
        print('=== fixed', fix_val, '===')
        l_vec = l.copy(); u_vec = u.copy(); l_vec[54] = fix_val; u_vec[54] = fix_val
    for scenario in scenarios:
        opt = simplinho.RevisedSimplexOptions()
        for key, value in scenario.items():
            setattr(opt, key, value)
        solver = simplinho.RevisedSimplex(opt)
        try:
            sol = solver.solve(A_sparse, b, c, l_vec, u_vec)
            print('scenario', scenario, 'status', sol.status, 'obj', sol.obj)
        except Exception as e:
            print('scenario', scenario, 'exception', type(e).__name__, e)


In [ ]:
import importlib
spec = importlib.util.spec_from_file_location("simplinho", module_path)
simplinho = importlib.util.module_from_spec(spec)
spec.loader.exec_module(simplinho)
n = 100
num_constraints = 4
total_vars = n + num_constraints
A = np.zeros((num_constraints, total_vars))
b = np.zeros(num_constraints)
c = np.zeros(total_vars)
l = np.zeros(total_vars)
u = np.full(total_vars, np.inf)
for i in range(n):
    A[0, i] = weights1[i]
    A[1, i] = weights2[i]
    A[2, i] = weights3[i]
    A[3, i] = groups[i]
b[0] = 0.35 * sum(weights1)
b[1] = 0.33 * sum(weights2)
b[2] = 0.31 * sum(weights3)
b[3] = max(8.0, 0.18 * n)
for row in range(num_constraints):
    A[row, n + row] = 1.0
c[:n] = -np.array(profits)
basis = [n + row for row in range(num_constraints)]
opt = simplinho.RevisedSimplexOptions()
opt.mode = simplinho.SimplexMode.Auto
opt.pricing_rule = 'adaptive'
solver = simplinho.RevisedSimplex(opt)
for fix_val in [None, 0.0, 1.0]:
    if fix_val is None:
        print('=== root with explicit basis ===')
        l_vec, u_vec = l, u
    else:
        print('=== fixed', fix_val, 'with explicit basis ===')
        l_vec = l.copy(); u_vec = u.copy(); l_vec[54] = fix_val; u_vec[54] = fix_val
    try:
        sol = solver.solve(A, b, c, l_vec, u_vec, basis)
        print('status', sol.status, 'obj', sol.obj)
    except Exception as e:
        print('exception', type(e).__name__, e)


In [ ]:
import importlib
spec = importlib.util.spec_from_file_location("simplinho", module_path)
simplinho = importlib.util.module_from_spec(spec)
spec.loader.exec_module(simplinho)
n = 100
profits = [float(((17 * i + 13) % 97) + 10) for i in range(n)]
weights1 = [float(((11 * i + 7) % 29) + 1) for i in range(n)]
weights2 = [float(((19 * i + 5) % 31) + 1) for i in range(n)]
weights3 = [float(((23 * i + 3) % 37) + 1) for i in range(n)]
groups = [1.0 if i % 10 in (0, 1, 2, 3) else 0.0 for i in range(n)]
model = simplinho.Model()
x_vars = [model.add_binary_var(f'x_{i}', obj=profits[i]) for i in range(n)]
for coeffs, sense, rhs, name in [
    (weights1, '<=', 0.35 * sum(weights1), 'cap_1'),
    (weights2, '<=', 0.33 * sum(weights2), 'cap_2'),
    (weights3, '<=', 0.31 * sum(weights3), 'cap_3'),
    (groups, '<=', max(8.0, 0.18 * n), 'group_cap'),
 ]:
    expr = 0.0
    for i, coeff in enumerate(coeffs):
        expr = expr + coeff * x_vars[i]
    if sense == '<=':
        model.add_constr(expr <= rhs, name=name)
    elif sense == '>=':
        model.add_constr(expr >= rhs, name=name)
    else:
        model.add_constr(expr == rhs, name=name)
model.maximize(sum(profits[i] * x_vars[i] for i in range(n)))
options = simplinho.BranchAndBoundOptions()
options.max_nodes = 10000
options.node_selection = simplinho.NodeSelectionStrategy.BestBound
options.parallel_workers = 1
options.use_async_heuristics = False
options.use_cut_pool = False
options.use_gomory_cuts = False
options.use_cover_cuts = False
options.use_feasibility_pump = False
options.use_rens = False
options.use_rins = False
options.use_local_search = False
options.use_local_branching = False
options.verbose = True
options.strong_branching_candidates = 0
options.branching_strategy = simplinho.BranchingStrategy.MostFractional
result = model.solve_mip(options)
print('Simplinho status:', result.status)
print('Objective:', result.obj)
print('Best bound:', result.best_bound)
print('Node count:', result.node_count)
print('Root relaxation objective:', result.root_relaxation_objective)
print('Has solution:', result.has_solution)
print('Warm start used:', result.warm_start_basis_state_used)


In [ ]:
import importlib
spec = importlib.util.spec_from_file_location("simplinho", module_path)
simplinho = importlib.util.module_from_spec(spec)
spec.loader.exec_module(simplinho)
n = 100
num_constraints = 4
total_vars = n + num_constraints
A = np.zeros((num_constraints, total_vars))
b = np.zeros(num_constraints)
c = np.zeros(total_vars)
l = np.zeros(total_vars)
u = np.full(total_vars, np.inf)
for i in range(n):
    A[0, i] = weights1[i]
    A[1, i] = weights2[i]
    A[2, i] = weights3[i]
    A[3, i] = groups[i]
b[0] = 0.35 * sum(weights1)
b[1] = 0.33 * sum(weights2)
b[2] = 0.31 * sum(weights3)
b[3] = max(8.0, 0.18 * n)
for row in range(num_constraints):
    A[row, n + row] = 1.0
c[:n] = -np.array(profits)
for mode in [simplinho.SimplexMode.Auto, simplinho.SimplexMode.Dual, simplinho.SimplexMode.Primal]:
    for pricing in ['adaptive', 'devex', 'most_negative']:
        opt = simplinho.RevisedSimplexOptions()
        opt.mode = mode
        opt.pricing_rule = pricing
        solver = simplinho.RevisedSimplex(opt)
        try:
            root = solver.solve(A, b, c, l, u)
            print('mode', mode, 'pricing', pricing, 'status', root.status, 'obj', root.obj)
        except Exception as e:
            print('mode', mode, 'pricing', pricing, 'error', type(e).__name__, e)


In [ ]:
import importlib
spec = importlib.util.spec_from_file_location("simplinho", module_path)
simplinho = importlib.util.module_from_spec(spec)
spec.loader.exec_module(simplinho)
n = 100
num_constraints = 4
total_vars = n + num_constraints
A = np.zeros((num_constraints, total_vars))
b = np.zeros(num_constraints)
c = np.zeros(total_vars)
l = np.zeros(total_vars)
u = np.full(total_vars, np.inf)
for i in range(n):
    A[0, i] = weights1[i]
    A[1, i] = weights2[i]
    A[2, i] = weights3[i]
    A[3, i] = groups[i]
b[0] = 0.35 * sum(weights1)
b[1] = 0.33 * sum(weights2)
b[2] = 0.31 * sum(weights3)
b[3] = max(8.0, 0.18 * n)
for row in range(num_constraints):
    A[row, n+row] = 1.0
c[:n] = -np.array(profits)
for mode in [simplinho.SimplexMode.Auto, simplinho.SimplexMode.Dual, simplinho.SimplexMode.Primal]:
    for pricing in ['adaptive', 'devex', 'most_negative']:`
        opt = simplinho.RevisedSimplexOptions()
        opt.mode = mode
        opt.pricing_rule = pricing
        solver = simplinho.RevisedSimplex(opt)
        try:
            root = solver.solve(A, b, c, l, u)
            print('mode', mode, 'pricing', pricing, 'status', root.status, 'obj', root.obj)
        except Exception as e:
            print('mode', mode, 'pricing', pricing, 'error', type(e).__name__, e)


In [ ]:
# Try low-level solves with explicit simplex options matching BnB default profile
opt = simplinho.RevisedSimplexOptions()
print('default mode', opt.mode, 'pricing', opt.pricing_rule)
for mode in [simplinho.SimplexMode.Auto, simplinho.SimplexMode.Dual, simplinho.SimplexMode.Primal]:
    opt.mode = mode
    solver = simplinho.RevisedSimplex(opt)
    try:
        root = solver.solve(A, b, c, l, u)
        print('mode', mode, 'status', root.status, 'obj', root.obj)
    except Exception as exc:
        print('mode', mode, 'exception', exc)


In [1]:
import importlib.util
from pathlib import Path
build_dir = Path('build')
module_path = next(build_dir.glob('simplinho*.so'), None)
assert module_path is not None, 'simplinho extension not found'
spec = importlib.util.spec_from_file_location('simplinho', module_path)
simplinho = importlib.util.module_from_spec(spec)
spec.loader.exec_module(simplinho)
print('loaded', module_path)
m = simplinho.Model()
x = m.add_binary_var('x', obj=1.0)
m.add_constr(x <= 1.0)
m.maximize(x)
res = m.solve_mip(simplinho.BranchAndBoundOptions())
print('status', res.status)
print('obj', res.obj)


loaded build/simplinho.cpython-313-x86_64-linux-gnu.so
status MIPStatus.Optimal
obj 1.0


In [2]:
# Execute the warm-start test script and print the output from the notebook kernel.
import subprocess
import sys
from pathlib import Path
script = Path('tmp_warm_start_test.py')
print('Running', script)
result = subprocess.run([sys.executable, '-u', str(script)], capture_output=True, text=True)
print('RETURN CODE', result.returncode)
print('STDOUT:\n', result.stdout)
print('STDERR:\n', result.stderr)

Running tmp_warm_start_test.py
RETURN CODE 0
STDOUT:
 Loaded simplinho from build
Solving initial LP...
[solve] start m=1 n=2
[presolve] actions=6 reduced_m=1 reduced_n=2 infeasible=0 unbounded=0
[presolve] #1 scale_row i=0 scale=1
[presolve] #2 scale_col j=0 scale=1
[presolve] #3 scale_col j=1 scale=1
[presolve] #4 row_reduce old_m=1 keep=1
[presolve] #5 tighten_bound j=0 old_l=0 old_u=inf
[presolve] #6 tighten_bound j=1 old_l=0 old_u=inf
[primal] start basis=[0]
[primal] iter=1 obj=-5 enter=1 leave_row=0 leave_var=0 step=5 alpha=1 basis_before=[0]
[primal] iter=1 basis_after=[1]
[primal] optimal iter=2 basis=[1]
Initial status: LPStatus.Optimal
Initial objective: -10.0
Initial primal: [0. 5.]
Initial basis: basis available, column_status length = 2
Updating bounds for x0 from [0, 10] to [1, 10].
Solving warm-started LP with previous basis...
[solve] start m=1 n=2
[solve] bound reformulation nv=2 upper_rows=0 total_m=1 total_n=2
[solve] start m=1 n=2
[presolve] actions=0 reduced_m=1 r